# ChatMoonshot examples

This notebook collects the main runnable examples used in the documentation for `langchain-moonshot`.

It defaults to the Moonshot China endpoint: `https://api.moonshot.cn/v1`.


## Setup

Before running the rest of the notebook, make sure `MOONSHOT_API_KEY` is available. If it is not already set, the next cell will prompt for it.


In [1]:
import base64
import getpass
import mimetypes
import os
from pathlib import Path

from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from pydantic import BaseModel, Field

from langchain_moonshot import ChatMoonshot

if not os.getenv("MOONSHOT_API_KEY"):
    os.environ["MOONSHOT_API_KEY"] = getpass.getpass("Enter your Moonshot API key: ")

os.environ.setdefault("MOONSHOT_API_BASE", "https://api.moonshot.cn/v1")

print("Using endpoint:", os.environ["MOONSHOT_API_BASE"])


Using endpoint: https://api.moonshot.cn/v1


In [2]:
def find_example_image() -> Path:
    candidates = [
        Path("image.png"),
        Path("examples/image.png"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find example image at image.png or examples/image.png"
    )


def to_data_url(path: Path) -> str:
    mime_type, _ = mimetypes.guess_type(path.name)
    if mime_type is None:
        mime_type = "image/png"
    data = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{data}"


## Instantiation and invocation


In [3]:
llm = ChatMoonshot(
    model="kimi-k2.5",
    thinking=False,
    temperature=0.6,
    max_retries=2,
)

messages = [
    ("system", "You are a concise bilingual assistant."),
    (
        "human",
        "Summarize why Moonshot reasoning models are useful in two bullet points.",
    ),
]

ai_msg = llm.invoke(messages)
print(ai_msg.text)
print(ai_msg.usage_metadata)


• **Enhanced problem-solving**: Moonshot reasoning models excel at complex, multi-step reasoning tasks (math, coding, logic) by breaking down problems and verifying intermediate steps.

• **Reduced errors**: They use techniques like chain-of-thought and self-correction to catch mistakes and produce more reliable, accurate outputs.
{'input_tokens': 32, 'output_tokens': 65, 'total_tokens': 97, 'input_token_details': {}, 'output_token_details': {}}


## Reasoning output


In [4]:
reasoning_llm = ChatMoonshot(
    model="kimi-k2.5",
    thinking=True,
    temperature=1.0,
)

reasoning_msg = reasoning_llm.invoke(
    "Explain in two bullet points why reasoning models are useful."
)

print(reasoning_msg.text)
print(reasoning_msg.additional_kwargs.get("reasoning_content"))


• **Superior complex problem-solving**: Reasoning models break down intricate tasks into explicit intermediate steps (chain-of-thought), enabling them to handle sophisticated challenges—such as mathematical proofs, debugging code, or multi-step logic—that overwhelm models attempting direct answers.

• **Transparency and error detection**: By exposing their internal reasoning process, these models allow users to audit the logic, spot where conclusions go wrong, and verify soundness, transforming opaque "black box" outputs into interpretable, trustworthy results.
The user wants an explanation of why reasoning models are useful, specifically in two bullet points. I need to provide a concise, clear explanation with exactly two bullet points covering the key benefits of reasoning models.

Key aspects of reasoning models (like chain-of-thought, step-by-step reasoning, etc.):
1. They break down complex problems into steps
2. They improve accuracy and reliability
3. They make the reasoning pro

## Streaming


In [5]:
streaming_llm = ChatMoonshot(
    model="kimi-k2.5",
    thinking=True,
    temperature=1.0,
    stream_usage=True,
)

full = None
for chunk in streaming_llm.stream(
    "Explain streaming output in two short bullet points."
):
    if full is None:
        full = chunk
    else:
        full += chunk

    if chunk.text:
        print(chunk.text, end="")

    reasoning = chunk.additional_kwargs.get("reasoning_content")
    if reasoning:
        print(f"\n[reasoning] {reasoning}", end="")

print()
print(full.usage_metadata if full is not None else None)



[reasoning] The
[reasoning]  user
[reasoning]  wants
[reasoning]  an
[reasoning]  explanation
[reasoning]  of
[reasoning]  "
[reasoning] streaming
[reasoning]  output
[reasoning] "
[reasoning]  in
[reasoning]  two
[reasoning]  short
[reasoning]  bullet
[reasoning]  points
[reasoning] .
[reasoning]  This
[reasoning]  is
[reasoning]  a
[reasoning]  concise
[reasoning]  request
[reasoning] ,
[reasoning]  likely
[reasoning]  referring
[reasoning]  to
[reasoning]  the
[reasoning]  concept
[reasoning]  in
[reasoning]  computing
[reasoning] /
[reasoning] AI
[reasoning]  where
[reasoning]  data
[reasoning]  is
[reasoning]  processed
[reasoning]  and
[reasoning]  sent
[reasoning]  in
[reasoning]  chunks
[reasoning]  rather
[reasoning]  than
[reasoning]  waiting
[reasoning]  for
[reasoning]  the
[reasoning]  complete
[reasoning]  result
[reasoning] .


[reasoning] Key
[reasoning]  points
[reasoning]  to
[reasoning]  cover
[reasoning] :

[reasoning] 1
[reasoning] .
[reasoning]  What
[reasoning] 

## Tool calling


In [6]:
@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


llm_with_tools = ChatMoonshot(
    model="kimi-k2.5",
    thinking=False,
    temperature=0.6,
).bind_tools([add, multiply])

tool_messages = [
    ("system", "Use the provided math tools before answering."),
    ("human", "Add 17 and 25, multiply 12 by 13, then summarize the results."),
]

tool_response = llm_with_tools.invoke(tool_messages)
print(tool_response.tool_calls)

if tool_response.tool_calls:
    tool_results = []
    for tool_call in tool_response.tool_calls:
        if tool_call["name"] == "add":
            result = add.invoke(tool_call["args"])
        else:
            result = multiply.invoke(tool_call["args"])
        tool_results.append(
            ToolMessage(content=str(result), tool_call_id=tool_call["id"])
        )

    final_response = llm_with_tools.invoke(
        [*tool_messages, tool_response, *tool_results]
    )
    print(final_response.text)


[{'name': 'add', 'args': {'a': 17, 'b': 25}, 'id': 'add:0', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 12, 'b': 13}, 'id': 'multiply:1', 'type': 'tool_call'}]
Here are the results:

- **Addition**: 17 + 25 = **42**
- **Multiplication**: 12 × 13 = **156**

To summarize: The sum of 17 and 25 is 42, and the product of 12 and 13 is 156.


## Structured output


In [7]:
class WeatherAnswer(BaseModel):
    city: str = Field(description="City name")
    summary: str = Field(description="One-sentence weather summary")


structured_llm = ChatMoonshot(
    model="kimi-k2.5",
    thinking=False,
    temperature=0.6,
).with_structured_output(WeatherAnswer)

structured_result = structured_llm.invoke("Summarize today's weather in Shanghai.")
print(structured_result)


city='Shanghai' summary='Today in Shanghai, expect partly cloudy skies with temperatures around 18-24°C, moderate humidity, and a gentle breeze from the southeast.'


## Multimodal image input

This cell uses the local example image shipped with the repository.


In [8]:
image_path = find_example_image()
image_data_url = to_data_url(image_path)
print("Using image:", image_path)

vision_llm = ChatMoonshot(
    model="moonshot-v1-32k-vision-preview",
)

vision_message = HumanMessage(
    content=[
        {"type": "text", "text": "Describe the image and mention one concrete detail."},
        {
            "type": "image_url",
            "image_url": {"url": image_data_url},
        },
    ]
)

vision_response = vision_llm.invoke([vision_message])
print(vision_response.text)


Using image: image.png
The image features a yellow frame with a picture of a book, and one detail is that the book has a green apple on the cover.
